In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.style as style
from statsmodels.tsa.api import seasonal_decompose
import datetime
import time
import pandas.util.testing as tm
import matplotlib.pyplot as mp
style.use('ggplot')

In [ ]:
# Intersecção e diferença
def inter_dif(lst1, lst2):
    intersection = set(lst1).intersection(lst2)
    difference1 = set(lst1).difference(set(lst2))
    difference2 = set(lst2).difference(set(lst1))
    print('total primeira: ', len(lst1))
    print('total segunda: ', len(lst2))
    print('intersecção: ', len(intersection))
    print('primeira - segunda: ', len(difference1))
    print('segunda - primeira', len(difference2))
    return intersection, difference1, difference2

In [ ]:
# INPUTS
#nome_atualizacao = 'nov23_late'
data = datetime.date(2023, 11, 22)
arquivo_produtos = "products_202311221520.csv"
arquivo_estoque_novos = "df_estoque_novos.csv"
arquivo_fornecedores_ampliado = 'df_fornecedoresNEW_20230425.csv'

dia_atual = data
#SET UP
dia = data.day
mes = data.month
ano = data.year
s_dia = '{:02d}'.format(data.day)
s_mes = '{:02d}'.format(data.month)
s_ano = str(ano)
s_data = s_ano+ s_mes + s_dia
dia_mes = dia

In [ ]:
str(data)

In [ ]:
# Reloading
with open(file = "df_adalbertoA.csv", mode = 'r') as f:
    df_adalbertoA = pd.read_csv(f, index_col=0)
with open(file = "df_adalbertoB.csv", mode = 'r') as f:
    df_adalbertoB = pd.read_csv(f, index_col=0)
with open(file = "df_adalbertoC.csv", mode = 'r') as f:
    df_adalbertoC = pd.read_csv(f, index_col=0)
list_adalbertoA = list(df_adalbertoA.CodProduto.unique())
list_adalbertoB = list(df_adalbertoB.CodProduto.unique())
list_adalbertoC = list(df_adalbertoC.CodProduto.unique())
list_ABC = list_adalbertoA + list_adalbertoB + list_adalbertoC
lista_ABC = list_ABC

# Reloading
with open(file = arquivo_produtos, mode = 'r') as f:
    df_produtos = pd.read_csv(f, index_col=0)

# Reloading
#with open(file = "forecast_202311071121.csv", mode = 'r') as f:
#    df_forecast = pd.read_csv(f, index_col=0)

# Reloading
with open(file = "df_sugestao_"+ s_data +"_c.csv", mode = 'r') as f:
    df_sugestao = pd.read_csv(f, index_col=0)

# Reloading
with open(file = arquivo_estoque_novos, mode = 'r') as f:
    df_estoque_novos = pd.read_csv(f, index_col=0)

# Reloading
df_fornecedores_ampliado = pd.read_csv(arquivo_fornecedores_ampliado)

In [ ]:
df_produtos[df_produtos.sugestao==-1]

In [ ]:
#gerar1

In [ ]:
df_gerar1 = df_produtos.copy()
df_gerar1 = df_gerar1[['sugestao']]

In [ ]:
df_gerar1.columns = ['sugestao_antigo']

In [ ]:
#gerar coluna igual ao id
#separá-la em duas listas
# Uma com os produtos a serem atualizados
# Outra com os produtos que já estão na sugestao
# Fazer dois replaces

In [ ]:
df_gerar1['sugestao']=df_gerar1.index

In [ ]:
df_gerar1

In [ ]:
len(list_ABC)

In [ ]:
list_produtos = list(df_produtos.codproduto)

In [ ]:
intersec = inter_dif(list_produtos, list_ABC)

In [ ]:
lista_trocar_id = df_produtos[df_produtos.codproduto.isin(list_ABC)].index
lista_trocar_value = df_produtos[df_produtos.codproduto.isin(list_ABC)].codproduto
df_gerar1b = df_gerar1[df_gerar1.sugestao.isin(lista_trocar_id)]

In [ ]:
len(lista_trocar_id)

In [ ]:
df_gerar1b = df_gerar1[df_gerar1.sugestao.isin(lista_trocar_id)]

In [ ]:
lista1 = lista_trocar_id
lista2 = lista_trocar_value
mapping = dict(zip(lista1, lista2))

df_gerar1b['sugestao']= df_gerar1b['sugestao'].map(mapping)

In [ ]:
df_gerar1b

In [ ]:
lista_cod = list(df_sugestao.index)
lista_sug = list(df_sugestao.Sugestao_Compra)
mapping = dict(zip(lista_cod, lista_sug))
df_gerar1b['sugestao'] = df_gerar1b['sugestao'].map(mapping)

In [ ]:
df_gerar1b

In [ ]:
df_produtos[df_produtos.codproduto=='00001903401']

In [ ]:
df_produtos[df_produtos.codproduto=='00003503502']

In [ ]:
df_gerar1b[df_gerar1b.index==373622]

In [ ]:
df_gerar1b[df_gerar1b.index==379657]

In [ ]:
bloblok = inter_dif(list(df_produtos[df_produtos.codproduto.isin(list_ABC)].index), list(df_gerar1b.index))

In [ ]:
# Salvando
df_gerar1b.to_csv(r'df_produtos_update_' + s_data + '.csv')

In [ ]:
df_sugestao[df_sugestao.Fornecedor=='VENCEDORA']

In [ ]:
df_sugestao = df_sugestao[df_sugestao.index.isin(lista_ABC)]

df1 = df_sugestao.copy()
df2 = df_sugestao.copy()
df3 = df_sugestao.copy()

df1 = df1.reset_index()
df2 = df2.reset_index()
df3 = df3.reset_index()

df1 = df1[['index','cobertura', 'h1']]
df2 = df2[['index','cobertura', 'h2']]
df3 = df3[['index','cobertura' ,'h3']]

df1.columns = ['codproduto','cobertura' , 'h1']
df2.columns = ['codproduto','cobertura' , 'h2']
df3.columns = ['codproduto','cobertura' , 'h3']

df_date = pd.DataFrame({'t':[dia_atual]*len(df1['h1'])})

df1['data']= pd.to_datetime(df_date['t']) + pd.to_timedelta(df1['cobertura'], unit='D')
df2['data']= pd.to_datetime(df_date['t']) + pd.to_timedelta(2*df1['cobertura'], unit='D')
df3['data']= pd.to_datetime(df_date['t']) + pd.to_timedelta(3*df1['cobertura'], unit='D')

df1.columns = ['codproduto', 'cobertura','quantidade', 'h']
df2.columns = ['codproduto', 'cobertura','quantidade', 'h']
df3.columns = ['codproduto', 'cobertura','quantidade', 'h']

df_forecast = pd.concat([df1, df2, df3], ignore_index=True)
df_forecast

df_forecast = df_forecast.reset_index()
df_forecast.columns = ['id', 'codproduto','cobertura', 'quantidade', 'h']
df_forecast.id = df_forecast.id + 1

list_produtos = list(df_produtos.codproduto)
list_produtos_id = list(df_produtos.index)
mapping = dict(zip(list_produtos, list_produtos_id))

df_forecast.codproduto = df_forecast.codproduto.map(mapping)

df_forecast.columns = ['id', 'id_prod','cobertura', 'quantidade', 'h']
#df_forecast.mes = pd.to_datetime(df_forecast.mes.astype(str), format='%Y-%m-%d')

#df_forecast = df_forecast[['id', 'id_prod', 'quantidade', 'h']]

# Salvando
#(df_forecast).to_csv(r'df_forecastABCNew_20230424.csv', index=False)

In [ ]:
df_forecast

In [ ]:
#temporario

In [ ]:
df_descricao_novos = df_estoque_novos[['codproduto','descricao', 'precoOriginal', 'desc_marca', 'armazempadr.', 'foradelinha']]
df_descricao_novos = df_descricao_novos.drop_duplicates(subset='codproduto')
display(df_descricao_novos)

lista_fornecedores_velhos = list(df_sugestao.Fornecedor.unique())
lista_fornecedores_novos = list(df_descricao_novos.desc_marca.unique())

kol = inter_dif(lista_fornecedores_velhos, lista_fornecedores_novos)
lista_fornecedores_diferenca = kol[2]

lista_produtos_novos = list(df_descricao_novos.codproduto)

df_produtos_add = pd.DataFrame({'id':lista_produtos_novos,
                                'cod':lista_produtos_novos,
                                'description':lista_produtos_novos,
                               'id_fornecedor':lista_produtos_novos,
                               'cobertura':lista_produtos_novos,
                               'profile':lista_produtos_novos,
                               'sugestao':lista_produtos_novos,
                               'armazem':lista_produtos_novos,
                               'customedio':lista_produtos_novos})
lista_descricao = list(df_descricao_novos.descricao)
mapping = dict(zip(lista_produtos_novos, lista_descricao))
df_produtos_add.description = df_produtos_add.description.map(mapping)
lista_customedio = list(df_descricao_novos.precoOriginal)
mapping = dict(zip(lista_produtos_novos, lista_customedio))
df_produtos_add.customedio = df_produtos_add.customedio.map(mapping)
lista_armazem = list(df_descricao_novos['armazempadr.'])
mapping = dict(zip(lista_produtos_novos, lista_armazem))
df_produtos_add.armazem = df_produtos_add.armazem.map(mapping)
lista_sugestao = [-1]*len(lista_produtos_novos)
mapping = dict(zip(lista_produtos_novos, lista_sugestao))
df_produtos_add.sugestao = df_produtos_add.sugestao.map(mapping)
lista_cobertura = [30]*len(lista_produtos_novos)
mapping = dict(zip(lista_produtos_novos, lista_cobertura))
df_produtos_add.cobertura = df_produtos_add.cobertura.map(mapping)
lista_fornecedor = list(df_descricao_novos.desc_marca)
mapping = dict(zip(lista_produtos_novos, lista_fornecedor))
df_produtos_add.id_fornecedor = df_produtos_add.id_fornecedor.map(mapping)



lista_forn = list(df_fornecedores_ampliado.fornecedor)
lista_forn_id = list(df_fornecedores_ampliado.id)
mapping = dict(zip(lista_forn, lista_forn_id))
df_produtos_add.id_fornecedor = df_produtos_add.id_fornecedor.map(mapping)
lista_profile = ['X']*len(lista_produtos_novos)
mapping = dict(zip(lista_produtos_novos, lista_profile))
df_produtos_add.profile = df_produtos_add.profile.map(mapping)
display(df_produtos_add)

drodro = df_produtos[df_produtos.codproduto.isin(list(df_descricao_novos.codproduto))]

list_prod = list(drodro.codproduto)
list_id = list(drodro.index)
mapping = dict(zip(list_prod, list_id))

df_produtos_add.id = df_produtos_add.id.map(mapping)
lista_total = list(df_produtos.codproduto.unique())
pluni = inter_dif(lista_total, lista_produtos_novos)
df_produtos_add_encontrados = df_produtos_add[df_produtos_add.cod.isin(pluni[0])]

# Salvando
df_produtos_add_encontrados.to_csv(r'df_produtos_add_encontrados.csv', index=False)

#df_produtos_novos_adicionados = pd.concat([df_produtos, df_produtos_add_encontrados], ignore_index=True)
#df_produtos_novos_adicionados.description = df_produtos_novos_adicionados.description.str.replace('"', 'pol')
#df_produtos_novos_adicionados.description = df_produtos_novos_adicionados.description.str.replace("'", 'pes')


In [ ]:
df_sugestao_add = pd.DataFrame({'cod': list(df_produtos_add_encontrados.cod),
                                'cobertura': list(df_produtos_add_encontrados.cobertura),
                                'h1': [-1]*len(df_produtos_add_encontrados),
                                'h2': [-1]*len(df_produtos_add_encontrados),
                                'h3': [-1]*len(df_produtos_add_encontrados)})

df1 = df_sugestao_add.copy()
df2 = df_sugestao_add.copy()
df3 = df_sugestao_add.copy()

df1 = df1[['cod','cobertura', 'h1']]
df2 = df2[['cod','cobertura', 'h2']]
df3 = df3[['cod','cobertura' ,'h3']]

df1.columns = ['codproduto','cobertura' , 'h1']
df2.columns = ['codproduto','cobertura' , 'h2']
df3.columns = ['codproduto','cobertura' , 'h3']

df_date = pd.DataFrame({'t':[dia_atual]*len(df1['h1'])})

df1['data']= pd.to_datetime(df_date['t']) + pd.to_timedelta(df1['cobertura'], unit='D')
df2['data']= pd.to_datetime(df_date['t']) + pd.to_timedelta(2*df1['cobertura'], unit='D')
df3['data']= pd.to_datetime(df_date['t']) + pd.to_timedelta(3*df1['cobertura'], unit='D')

df1.columns = ['codproduto', 'cobertura','quantidade', 'h']
df2.columns = ['codproduto', 'cobertura','quantidade', 'h']
df3.columns = ['codproduto', 'cobertura','quantidade', 'h']

df_forecast_add = pd.concat([df1, df2, df3], ignore_index=True)
df_forecast_add

df_forecast_add = df_forecast_add.reset_index()
df_forecast_add.columns = ['id', 'codproduto','cobertura', 'quantidade', 'h']
df_forecast_add.id = df_forecast_add.id + 1

list_produtos = list(df_produtos_add_encontrados.cod)
list_produtos_id = list(df_produtos_add_encontrados.id)
mapping = dict(zip(list_produtos, list_produtos_id))
df_forecast_add.codproduto = df_forecast_add.codproduto.map(mapping)
df_forecast_add.columns = ['id', 'id_prod','cobertura', 'quantidade', 'h']
#df_forecast.mes = pd.to_datetime(df_forecast.mes.astype(str), format='%Y-%m-%d')

#df_forecast = df_forecast[['id', 'id_prod', 'quantidade', 'h']]

df_forecast_add_encontrados = pd.concat([df_forecast, df_forecast_add], ignore_index=True)
df_forecast_add_encontrados = df_forecast_add_encontrados.iloc[:, 1:]
df_forecast_add_encontrados = df_forecast_add_encontrados.reset_index()
df_forecast_add_encontrados.columns = ['id', 'id_prod', 'cobertura', 'quantidade', 'h']
df_forecast_add_encontrados.id = df_forecast_add_encontrados.id + 1

display(df_forecast_add_encontrados)
# Salvando
df_forecast_add_encontrados.to_csv(r'df_forecastNew_' + s_data +'.csv', index=False)